In [1]:
from openvino import Core
core = Core()
print(core.available_devices)
# Should print: ['CPU', 'GPU', 'NPU']


['CPU', 'GPU', 'NPU']


In [ ]:
from optimum.intel import OVModelForCausalLM
from transformers import AutoTokenizer
import openvino as ov

model_path = "../models/phi3.5-mini-ov-int4"

# 1. Load without compiling
model = OVModelForCausalLM.from_pretrained(
    model_path,
    device="GPU",
    compile=False,
    ov_config={"CACHE_DIR": "./ov_cache"}
)

# 2. MANUALLY SET INPUT TYPES TO I32
#    This modifies the OpenVINO graph in-memory to expect 32-bit integers
#    instead of 64-bit, satisfying the NPU requirement.
for item in model.model.inputs:
    if item.get_element_type() == ov.Type.i64:
        item.get_node().set_element_type(ov.Type.i32)
        # Also need to ensure the tensor names/layouts align if this was more complex,
        # but for changing precision usually this is enough or we use PrePostProcessor.

# Alternative: Use PrePostProcessor if the direct set fails (which it might on frozen nodes)
# But let's try the reshape+compile first, as sometimes reshape triggers type recalculation if we are lucky.

# 3. Resize and Compile
model.reshape(1, 128)
model.compile()

tokenizer = AutoTokenizer.from_pretrained(model_path)
input_text = "Explain transformer attention:"
inputs = tokenizer(input_text, return_tensors="pt")

# IMPORTANT: Cast PyTorch inputs to int32 before generating!
inputs = {k: v.to(dtype=int) for k, v in inputs.items()} # .to(int) in PyTorch is int32 usually

outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Static shapes are not supported for causal language model.


RuntimeError: Exception from src\inference\src\cpp\core.cpp:107:
Exception from src\inference\src\dev\plugin.cpp:53:
Exception from src\plugins\intel_npu\src\plugin\src\plugin.cpp:717:
Exception from src\plugins\intel_npu\src\compiler_adapter\src\ze_graph_ext_wrappers.cpp:389:
L0 pfnCreate2 result: ZE_RESULT_ERROR_INVALID_ARGUMENT, code 0x78000004 - generic error code for invalid arguments . [NPU_VCL] Compiler returned msg:
Exception from src\core\src\partial_shape.cpp:266:
to_shape was called on a dynamic shape.




